# Cross-sensory Uni-Mol LoRA — Colab

This notebook runs only in a Colab GPU runtime. It uses curated ChemTastesDB labels for the main taste task and keeps FlavorDB taste wording as weak auxiliary evidence.

Before running: push/upload the current project version containing `src/dataset/sensory.py`, `src/sensory/`, and `scripts/train_cross_sensory.py`.

In [ ]:
%pip install -q unimol-tools pandas pyarrow scikit-learn rdkit openpyxl

import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

## Get the current project

Replace `REPO_URL` if needed. The upstream repository must include the new cross-sensory files; cloning an older commit will fail the guard below.

In [ ]:
from pathlib import Path
import subprocess, sys

REPO_URL = 'https://github.com/FufanLu/molecular-foundation-model.git'  # replace with your pushed branch/repository
repo_dir = Path('/content/molecular-foundation-model')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(repo_dir)], check=True)
sys.path.insert(0, str(repo_dir))
required_code = [repo_dir / 'src/dataset/sensory.py', repo_dir / 'src/sensory/model.py', repo_dir / 'scripts/train_cross_sensory.py']
missing_code = [str(path) for path in required_code if not path.exists()]
if missing_code:
    raise RuntimeError('This clone is missing the cross-sensory implementation. Push/upload the current project first: ' + ', '.join(missing_code))
%cd /content/molecular-foundation-model

## Upload the four raw files

This cell opens Colab's file picker only when a required file is absent. Select all four files together: `leffingwell_merged.csv`, `goodscents_merged.csv`, `flavordb_merged.csv`, and `ChemTastesDB_database.xlsx`.

In [ ]:
from google.colab import files
import shutil

raw_dir = Path('data/raw/leffingwell')
raw_dir.mkdir(parents=True, exist_ok=True)
required_data = {'leffingwell_merged.csv', 'goodscents_merged.csv', 'flavordb_merged.csv', 'ChemTastesDB_database.xlsx'}
missing_data = required_data - {path.name for path in raw_dir.iterdir()}
if missing_data:
    print('Upload:', sorted(missing_data))
    uploaded = files.upload()
    for filename in uploaded:
        if filename in required_data:
            shutil.move(filename, raw_dir / filename)
missing_data = required_data - {path.name for path in raw_dir.iterdir()}
if missing_data:
    raise FileNotFoundError(f'Missing required raw files: {sorted(missing_data)}')
print('Raw data ready:', sorted(required_data))

## Build the strong/weak sensory dataset

`taste_labels` are ChemTastesDB-curated labels. FlavorDB-derived terms remain in `taste_weak_labels`; they supervise a separate low-weight weak-flavor head and weak-only high-temperature contrastive guidance.

In [ ]:
!python -m src.dataset.sensory --raw-dir data/raw/leffingwell --chem-tastes data/raw/leffingwell/ChemTastesDB_database.xlsx --output-dir data/processed/sensory --n-splits 5

import json
audit = json.loads(Path('data/processed/sensory/audit.json').read_text())
print('strong taste labels:', audit['taste_label_counts'])
print('exact odor–taste pairs:', audit['paired_molecule_count'])
print('salty examples:', audit['salty_molecule_count'])
print('fold summary:', json.dumps(audit['fold_summary'], indent=2))

## Train one fold first

This uses fold 0 as test, fold 1 as validation, and the remaining folds for training. The sampler places at least two exact pairs in every training batch so the InfoNCE term is non-zero. Checkpoints and JSON metrics are written under `outputs/cross_sensory/`.

In [ ]:
!PYTHONPATH=/content/molecular-foundation-model:$PYTHONPATH python scripts/train_cross_sensory.py --data data/processed/sensory/molecules.parquet --output-dir outputs/cross_sensory_weak_lora4_w002 --test-fold 0 --val-fold 1 --epochs 30 --patience 6 --batch-size 16 --lora-layers 4 --lora-rank 4 --paired-per-batch 2 --weak-paired-per-batch 2 --contrastive-weight 0.05 --weak-taste-weight 0.02 --weak-contrastive-weight 0 --weak-temperature 0.2

After fold 0 is stable, run the same command four more times with distinct `(test-fold, val-fold)` pairs. Do not combine salty into the main four-label taste macro-F1; it remains a separate low-shot evaluation.